In [ ]:
import pandas as pd

# 数据导入
ob = pd.read_csv('/Users/wook/Desktop/因子挖掘/raw_data/BTCUSDT_book_snapshot_5_20240528.csv.gz')
tr = pd.read_csv('/Users/wook/Desktop/因子挖掘/raw_data/BTCUSDT_trades_20240528.csv.gz')

### 处理原始数据到1s

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from datetime import datetime
from tqdm import tqdm

def process_orderbook_data(df, pbar=None):
    """
    处理订单簿数据的降采样
    """
    if pbar:
        pbar.set_description("转换时间戳")
    # 转换时间戳为datetime
    df['datetime'] = pd.to_datetime(df['timestamp'], unit='us')
    
    if pbar:
        pbar.set_description("设置索引")
    # 设置时间为索引
    df.set_index('datetime', inplace=True)
    
    if pbar:
        pbar.set_description("重采样处理")
    # 按100ms重采样，对于订单簿数据，我们取每个时间窗口的最后一个值
    # 因为订单簿是状态数据，最后一个值代表该时间窗口结束时的状态
    resampled = df.resample('1s').last()
    
    if pbar:
        pbar.set_description("清理数据")
    # 删除空行（没有数据的时间窗口）
    resampled.dropna(inplace=True)
    
    return resampled

def process_trades_data(df, pbar=None):
    """
    处理交易数据的降采样
    """
    if pbar:
        pbar.set_description("转换时间戳")
    # 转换时间戳为datetime
    df['datetime'] = pd.to_datetime(df['timestamp'], unit='us')
    
    if pbar:
        pbar.set_description("设置索引")
    # 设置时间为索引
    df.set_index('datetime', inplace=True)
    
    if pbar:
        pbar.set_description("基础重采样")
    # 按100ms重采样，对交易数据进行聚合
    agg_dict = {
        'exchange': 'first',
        'symbol': 'first',
        'timestamp': 'last',
        'local_timestamp': 'last',
        'id': 'last',
        'side': lambda x: x.value_counts().index[0] if len(x) > 0 else None,  # 主导方向
        'price': 'last',  # 最后成交价
        'amount': 'sum'   # 成交量求和
    }
    
    resampled = df.resample('1s').agg(agg_dict)
    
    if pbar:
        pbar.set_description("计算统计信息")
    # 删除空行
    resampled.dropna(inplace=True)
    
    # 添加一些统计信息
    df_temp = df.copy()
    
    # 计算每个100ms窗口的其他统计信息
    stats = df_temp.resample('1s').agg({
        'price': ['first', 'min', 'max', 'mean'],
        'amount': ['count', 'sum'],
        'side': lambda x: (x == 'buy').sum()  # 买单数量
    })
    
    if pbar:
        pbar.set_description("合并数据")
    # 展平多级列名
    stats.columns = ['_'.join(col).strip() for col in stats.columns]
    
    # 合并主要数据和统计数据
    resampled = resampled.join(stats, how='inner')
    
    # 重命名列
    resampled.rename(columns={
        'price_first': 'open_price',
        'price_min': 'min_price', 
        'price_max': 'max_price',
        'price_mean': 'avg_price',
        'amount_count': 'trade_count',
        'amount_sum': 'total_volume',
        'side_<lambda>': 'buy_count'
    }, inplace=True)
    
    # 计算卖单数量
    resampled['sell_count'] = resampled['trade_count'] - resampled['buy_count']
    
    return resampled

def downsample_files(data_dir='raw_data', output_dir='processed_data'):
    """
    批量处理文件夹中的所有数据文件
    """
    # 创建输出目录
    os.makedirs(output_dir, exist_ok=True)
    
    # 查找所有csv.gz文件
    pattern = os.path.join(data_dir, '*.csv.gz')
    files = glob.glob(pattern)
    
    print(f"找到 {len(files)} 个文件待处理")
    
    # 创建总体进度条
    with tqdm(total=len(files), desc="处理文件", unit="文件") as main_pbar:
        for file_path in files:
            try:
                filename = os.path.basename(file_path)
                main_pbar.set_description(f"处理: {filename}")
                
                # 创建单个文件的进度条
                with tqdm(total=5, desc="读取数据", leave=False) as file_pbar:
                    # 读取数据
                    file_pbar.set_description("读取数据文件")
                    df = pd.read_csv(file_path, compression='gzip')
                    file_pbar.update(1)
                    
                    # 根据文件名判断数据类型
                    if 'book_snapshot_5' in filename:
                        # 订单簿数据
                        file_pbar.set_description("处理订单簿数据")
                        processed_df = process_orderbook_data(df.copy(), file_pbar)
                        output_filename = filename.replace('.csv.gz', '_1s_orderbook.csv')
                        file_pbar.update(1)
                        
                    elif 'trades' in filename:
                        # 交易数据
                        file_pbar.set_description("处理交易数据")
                        processed_df = process_trades_data(df.copy(), file_pbar)
                        output_filename = filename.replace('.csv.gz', '_1s_trades.csv')
                        file_pbar.update(1)
                        
                    else:
                        print(f"跳过未知类型文件: {filename}")
                        main_pbar.update(1)
                        continue
                    
                    # 保存处理后的数据
                    file_pbar.set_description("保存文件")
                    output_path = os.path.join(output_dir, output_filename)
                    processed_df.to_csv(output_path)
                    file_pbar.update(1)
                    
                    file_pbar.set_description("完成")
                    file_pbar.update(1)
                
                tqdm.write(f"✓ 已完成: {output_filename}")
                tqdm.write(f"  原始数据行数: {len(df):,}, 降采样后行数: {len(processed_df):,}")
                tqdm.write("-" * 50)
                
            except Exception as e:
                tqdm.write(f"✗ 处理文件 {file_path} 时出错: {str(e)}")
                continue
            finally:
                main_pbar.update(1)
    
    print("\n🎉 批量处理完成!")

def process_single_file(file_path, output_dir='processed_data_10s'):
    """
    处理单个文件
    """
    os.makedirs(output_dir, exist_ok=True)
    
    filename = os.path.basename(file_path)
    print(f"正在处理: {filename}")
    
    with tqdm(total=4, desc="处理步骤") as pbar:
        # 读取数据
        pbar.set_description("读取数据")
        df = pd.read_csv(file_path, compression='gzip')
        pbar.update(1)
        
        # 根据文件名判断数据类型
        if 'book_snapshot_5' in filename:
            # 订单簿数据
            pbar.set_description("处理订单簿数据")
            processed_df = process_orderbook_data(df.copy(), pbar)
            output_filename = filename.replace('.csv.gz', '_10s_orderbook.csv')
            
            print("✓ 订单簿数据处理完成")
            print(f"  包含价格档位: asks[0-4], bids[0-4]")
            
        elif 'trades' in filename:
            # 交易数据
            pbar.set_description("处理交易数据")
            processed_df = process_trades_data(df.copy(), pbar)
            output_filename = filename.replace('.csv.gz', '_10s_trades.csv')
            
            print("✓ 交易数据处理完成")
            print("  新增统计字段: open_price, min_price, max_price, avg_price, trade_count, buy_count, sell_count")
            
        else:
            raise ValueError(f"未知的文件类型: {filename}")
        
        pbar.update(1)
        
        # 保存处理后的数据
        pbar.set_description("保存文件")
        output_path = os.path.join(output_dir, output_filename)
        processed_df.to_csv(output_path)
        pbar.update(1)
        
        pbar.set_description("完成")
        pbar.update(1)
    
    print(f"✓ 已保存到: {output_path}")
    print(f"  原始数据行数: {len(df):,}, 降采样后行数: {len(processed_df):,}")
    
    return processed_df

# 使用示例
if __name__ == "__main__":
    # 方法1: 批量处理所有文件
    print("开始批量处理所有文件...")
    downsample_files('raw_data', 'processed_data_1s')
    
    # 方法2: 处理单个文件（可选）
    # single_file = 'raw_data/BTCUSDTbook_snapshot_520240528.csv.gz'
    # if os.path.exists(single_file):
    #     process_single_file(single_file, 'processed_data')

### 处理筛选数据到1s

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from tqdm import tqdm

def process_orderbook_data(df, pbar=None):
    """
    处理订单簿数据的降采样
    """
    if pbar:
        pbar.set_description("转换时间戳")
    # 转换时间戳为datetime
    df['datetime'] = pd.to_datetime(df['timestamp'], unit='us')
    
    if pbar:
        pbar.set_description("设置索引")
    # 设置时间为索引
    df.set_index('datetime', inplace=True)
    
    if pbar:
        pbar.set_description("重采样处理")
    # 按1s重采样，对于订单簿数据，我们取每个时间窗口的最后一个值
    # 因为订单簿是状态数据，最后一个值代表该时间窗口结束时的状态
    resampled = df.resample('1s').last()
    
    if pbar:
        pbar.set_description("清理数据")
    # 删除空行（没有数据的时间窗口）
    resampled.dropna(inplace=True)
    
    return resampled

def downsample_files(input_dir, output_dir):
    """
    批量处理文件夹中的所有数据文件
    """
    # 创建输出目录
    os.makedirs(output_dir, exist_ok=True)
    
    # 查找所有csv文件
    pattern = os.path.join(input_dir, '*.csv')
    files = glob.glob(pattern)
    
    if not files:
        print(f"在路径 {input_dir} 中没有找到CSV文件")
        return
    
    print(f"找到 {len(files)} 个文件待处理")
    
    # 创建总体进度条
    with tqdm(total=len(files), desc="处理文件", unit="文件") as main_pbar:
        for file_path in files:
            try:
                filename = os.path.basename(file_path)
                main_pbar.set_description(f"处理: {filename}")
                
                # 创建单个文件的进度条
                with tqdm(total=4, desc="读取数据", leave=False) as file_pbar:
                    # 读取数据
                    file_pbar.set_description("读取数据文件")
                    df = pd.read_csv(file_path)
                    file_pbar.update(1)
                    
                    # 处理订单簿数据
                    file_pbar.set_description("处理订单簿数据")
                    processed_df = process_orderbook_data(df.copy(), file_pbar)
                    output_filename = filename  # 保持原文件名
                    file_pbar.update(1)
                    
                    # 保存处理后的数据
                    file_pbar.set_description("保存文件")
                    output_path = os.path.join(output_dir, output_filename)
                    processed_df.to_csv(output_path)
                    file_pbar.update(1)
                    
                    file_pbar.set_description("完成")
                    file_pbar.update(1)
                
                tqdm.write(f"✓ 已完成: {output_filename}")
                
            except Exception as e:
                tqdm.write(f"✗ 处理文件 {file_path} 时出错: {str(e)}")
                continue
            finally:
                main_pbar.update(1)
    
    print("\n🎉 批量处理完成!")

def process_single_file(file_path, output_dir):
    """
    处理单个文件
    """
    os.makedirs(output_dir, exist_ok=True)
    
    filename = os.path.basename(file_path)
    print(f"正在处理: {filename}")
    
    with tqdm(total=4, desc="处理步骤") as pbar:
        # 读取数据
        pbar.set_description("读取数据")
        df = pd.read_csv(file_path)
        pbar.update(1)
        
        # 处理订单簿数据
        pbar.set_description("处理订单簿数据")
        processed_df = process_orderbook_data(df.copy(), pbar)
        output_filename = filename  # 保持原文件名
        pbar.update(1)
        
        # 保存处理后的数据
        pbar.set_description("保存文件")
        output_path = os.path.join(output_dir, output_filename)
        processed_df.to_csv(output_path)
        pbar.update(1)
        
        pbar.set_description("完成")
        pbar.update(1)
    
    print(f"✓ 已保存到: {output_path}")
    
    return processed_df

# 使用示例
if __name__ == "__main__":
    # 定义输入和输出路径
    input_dir = "/Users/wook/Downloads/因子挖掘/Sample data/raw/筛选BTC分段数据/book25_snapshot"
    output_dir = "/Users/wook/Downloads/因子挖掘/Sample data/raw/筛选BTC分段数据/book25_snapshot_1s"
    
    # 批量处理所有文件
    print("开始批量处理所有文件...")
    downsample_files(input_dir, output_dir)

### 处理筛选数据H5版

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from tqdm import tqdm

def process_orderbook_data(df, pbar=None):
    """
    处理订单簿数据的降采样
    """
    if pbar:
        pbar.set_description("转换时间戳")
    # 转换时间戳为datetime
    df['datetime'] = pd.to_datetime(df['timestamp'], unit='us')
    
    if pbar:
        pbar.set_description("设置索引")
    # 设置时间为索引
    df.set_index('datetime', inplace=True)
    
    if pbar:
        pbar.set_description("重采样处理")
    # 按1s重采样，对于订单簿数据，我们取每个时间窗口的最后一个值
    # 因为订单簿是状态数据，最后一个值代表该时间窗口结束时的状态
    resampled = df.resample('1s').last()
    
    if pbar:
        pbar.set_description("清理数据")
    # 删除空行（没有数据的时间窗口）
    resampled.dropna(inplace=True)
    
    return resampled

def downsample_h5_files(input_dir, output_dir):
    """
    批量处理文件夹中的所有H5数据文件
    """
    # 创建输出目录
    os.makedirs(output_dir, exist_ok=True)
    
    # 查找所有h5文件
    pattern = os.path.join(input_dir, '*.h5')
    files = glob.glob(pattern)
    
    if not files:
        print(f"在路径 {input_dir} 中没有找到H5文件")
        return
    
    print(f"找到 {len(files)} 个H5文件待处理")
    
    # 创建总体进度条
    with tqdm(total=len(files), desc="处理文件", unit="文件") as main_pbar:
        for file_path in files:
            try:
                filename = os.path.basename(file_path)
                main_pbar.set_description(f"处理: {filename}")
                
                # 创建单个文件的进度条
                with tqdm(total=4, desc="读取数据", leave=False) as file_pbar:
                    # 读取H5数据
                    file_pbar.set_description("读取H5数据文件")
                    df = pd.read_hdf(file_path, key='snapshot')
                    file_pbar.update(1)
                    
                    # 处理订单簿数据
                    file_pbar.set_description("处理订单簿数据")
                    processed_df = process_orderbook_data(df.copy(), file_pbar)
                    file_pbar.update(1)
                    
                    # 保存处理后的数据为H5格式
                    file_pbar.set_description("保存H5文件")
                    output_path = os.path.join(output_dir, filename)
                    processed_df.to_hdf(output_path, key='snapshot', mode='w', format='table', complevel=9)
                    file_pbar.update(1)
                    
                    file_pbar.set_description("完成")
                    file_pbar.update(1)
                
                tqdm.write(f"✓ 已完成: {filename}")
                
            except Exception as e:
                tqdm.write(f"✗ 处理文件 {file_path} 时出错: {str(e)}")
                continue
            finally:
                main_pbar.update(1)
    
    print("\n🎉 批量处理完成!")

def process_single_h5_file(file_path, output_dir):
    """
    处理单个H5文件
    """
    os.makedirs(output_dir, exist_ok=True)
    
    filename = os.path.basename(file_path)
    print(f"正在处理: {filename}")
    
    with tqdm(total=4, desc="处理步骤") as pbar:
        # 读取H5数据
        pbar.set_description("读取H5数据")
        df = pd.read_hdf(file_path, key='snapshot')
        pbar.update(1)
        
        # 处理订单簿数据
        pbar.set_description("处理订单簿数据")
        processed_df = process_orderbook_data(df.copy(), pbar)
        pbar.update(1)
        
        # 保存处理后的数据为H5格式
        pbar.set_description("保存H5文件")
        output_path = os.path.join(output_dir, filename)
        processed_df.to_hdf(output_path, key='snapshot', mode='w', format='table', complevel=9)
        pbar.update(1)
        
        pbar.set_description("完成")
        pbar.update(1)
    
    print(f"✓ 已保存到: {output_path}")
    
    return processed_df

# 使用示例
if __name__ == "__main__":
    # 定义输入和输出路径
    input_dir = "/Users/wook/Downloads/因子挖掘/Sample data/raw/筛选BTC分段数据/book25_snapshot"
    output_dir = "/Users/wook/Downloads/因子挖掘/Sample data/raw/筛选BTC分段数据/book25_snapshot_1s"
    
    # 批量处理所有H5文件
    print("开始批量处理所有H5文件...")
    downsample_h5_files(input_dir, output_dir)

### 完整数据降频

In [2]:
import pandas as pd
import numpy as np
import os
import glob
import gzip
from tqdm import tqdm
import re

def read_csv_gz(file_path):
    """
    读取.csv.gz文件
    """
    try:
        with gzip.open(file_path, 'rt', encoding='utf-8') as f:
            df = pd.read_csv(f)
        return df
    except Exception as e:
        print(f"读取文件 {file_path} 时出错: {str(e)}")
        return None

def process_orderbook_data(df, pbar=None):
    """
    处理订单簿数据的降采样到1秒
    """
    if pbar:
        pbar.set_description("转换时间戳")
    
    # 根据实际列名调整，常见的时间戳列名
    timestamp_col = None
    for col in ['timestamp', 'time', 'datetime', 'ts', 'T']:
        if col in df.columns:
            timestamp_col = col
            break
    
    if timestamp_col is None:
        # 如果没有找到明确的时间戳列，使用第一列
        timestamp_col = df.columns[0]
        print(f"未找到标准时间戳列，使用第一列: {timestamp_col}")
    
    # 转换时间戳为datetime
    if df[timestamp_col].dtype in ['int64', 'float64']:
        # 检查时间戳的数量级来判断单位
        sample_timestamp = df[timestamp_col].iloc[0]
        if sample_timestamp > 1e15:  # 微秒 (1677721600000000)
            df['datetime'] = pd.to_datetime(df[timestamp_col], unit='us')
        elif sample_timestamp > 1e12:  # 毫秒 (1677721600000)
            df['datetime'] = pd.to_datetime(df[timestamp_col], unit='ms')
        elif sample_timestamp > 1e9:   # 秒 (1677721600)
            df['datetime'] = pd.to_datetime(df[timestamp_col], unit='s')
        else:  # 纳秒或其他
            df['datetime'] = pd.to_datetime(df[timestamp_col], unit='ns')
    else:
        # 如果已经是字符串格式的时间
        df['datetime'] = pd.to_datetime(df[timestamp_col])
    
    if pbar:
        pbar.set_description("设置索引")
    
    # 设置时间为索引
    df.set_index('datetime', inplace=True)
    
    if pbar:
        pbar.set_description("重采样处理")
    
    # 按1s重采样，对于订单簿数据，我们取每个时间窗口的最后一个值
    # 因为订单簿是状态数据，最后一个值代表该时间窗口结束时的状态
    resampled = df.resample('1s').last()
    
    if pbar:
        pbar.set_description("清理数据")
    
    # 删除空行（没有数据的时间窗口）
    resampled.dropna(inplace=True)
    
    return resampled

def find_book_files(raw_data_dir):
    """
    查找所有订单簿数据文件
    """
    # 查找所有订单簿文件
    pattern = os.path.join(raw_data_dir, "*book_snapshot*.csv.gz")
    book_files = glob.glob(pattern)
    
    if not book_files:
        print(f"在路径 {raw_data_dir} 中没有找到订单簿文件")
        return []
    
    return sorted(book_files)

def process_book_files_to_h5(raw_data_dir, output_dir):
    """
    批量处理订单簿文件并保存为H5格式
    """
    # 创建输出目录
    os.makedirs(output_dir, exist_ok=True)
    
    # 查找所有订单簿文件
    book_files = find_book_files(raw_data_dir)
    
    if not book_files:
        return
    
    print(f"找到 {len(book_files)} 个订单簿文件待处理")
    
    # 创建总体进度条
    with tqdm(total=len(book_files), desc="处理订单簿文件", unit="文件") as main_pbar:
        for book_file in book_files:
            try:
                book_basename = os.path.basename(book_file)
                main_pbar.set_description(f"处理: {book_basename}")
                
                # 创建单个文件的进度条
                with tqdm(total=4, desc="处理步骤", leave=False) as file_pbar:
                    
                    # 读取订单簿数据
                    file_pbar.set_description("读取订单簿数据")
                    book_df = read_csv_gz(book_file)
                    if book_df is None:
                        continue
                    file_pbar.update(1)
                    
                    # 显示原始数据信息
                    tqdm.write(f"原始数据形状: {book_df.shape}")
                    tqdm.write(f"时间范围: {book_df.iloc[0, 0]} - {book_df.iloc[-1, 0]}")
                    
                    # 处理订单簿数据（降频到1s）
                    file_pbar.set_description("处理订单簿数据")
                    processed_book_df = process_orderbook_data(book_df.copy(), file_pbar)
                    file_pbar.update(1)
                    
                    # 显示处理后数据信息
                    tqdm.write(f"降频后数据形状: {processed_book_df.shape}")
                    
                    # 保存为H5格式
                    file_pbar.set_description("保存H5文件")
                    h5_filename = book_basename.replace('.csv.gz', '_1s.h5')
                    h5_output_path = os.path.join(output_dir, h5_filename)
                    
                    # 保存为H5格式，使用压缩
                    processed_book_df.to_hdf(h5_output_path, 
                                           key='snapshot', 
                                           mode='w', 
                                           format='table', 
                                           complevel=9,
                                           complib='zlib')
                    file_pbar.update(1)
                    
                    file_pbar.set_description("完成")
                    file_pbar.update(1)
                
                tqdm.write(f"✓ 已完成: {book_basename}")
                tqdm.write(f"  保存至: {h5_output_path}")
                tqdm.write(f"  数据压缩比: {book_df.shape[0]} -> {processed_book_df.shape[0]} 行")
                tqdm.write("")
                
            except Exception as e:
                tqdm.write(f"✗ 处理文件 {book_file} 时出错: {str(e)}")
                continue
            finally:
                main_pbar.update(1)
    
    print("🎉 批量处理完成!")

def process_single_book_file(book_file, output_dir):
    """
    处理单个订单簿文件
    """
    os.makedirs(output_dir, exist_ok=True)
    
    book_basename = os.path.basename(book_file)
    print(f"正在处理订单簿文件: {book_basename}")
    
    with tqdm(total=4, desc="处理步骤") as pbar:
        # 读取订单簿数据
        pbar.set_description("读取订单簿数据")
        book_df = read_csv_gz(book_file)
        if book_df is None:
            return None
        pbar.update(1)
        
        print(f"原始数据形状: {book_df.shape}")
        print(f"列名: {list(book_df.columns)}")
        
        # 处理订单簿数据
        pbar.set_description("处理订单簿数据")
        processed_book_df = process_orderbook_data(book_df.copy(), pbar)
        pbar.update(1)
        
        print(f"降频后数据形状: {processed_book_df.shape}")
        
        # 保存为H5格式
        pbar.set_description("保存H5文件")
        h5_filename = book_basename.replace('.csv.gz', '_1s.h5')
        h5_output_path = os.path.join(output_dir, h5_filename)
        
        processed_book_df.to_hdf(h5_output_path, 
                               key='snapshot', 
                               mode='w', 
                               format='table', 
                               complevel=9,
                               complib='zlib')
        pbar.update(1)
        
        pbar.set_description("完成")
        pbar.update(1)
    
    print(f"✓ 已保存到: {h5_output_path}")
    print(f"✓ 数据压缩比: {book_df.shape[0]} -> {processed_book_df.shape[0]} 行")
    
    return processed_book_df

def inspect_book_data(book_file, n_rows=5):
    """
    检查订单簿数据的结构
    """
    print(f"检查文件: {os.path.basename(book_file)}")
    
    book_df = read_csv_gz(book_file)
    if book_df is None:
        return
    
    print(f"数据形状: {book_df.shape}")
    print(f"列名: {list(book_df.columns)}")
    print(f"数据类型:")
    print(book_df.dtypes)
    print(f"\n前{n_rows}行数据:")
    print(book_df.head(n_rows))
    print(f"\n数据统计:")
    print(book_df.describe())

# 使用示例
if __name__ == "__main__":
    # 定义路径
    raw_data_dir = "/Users/wook/Downloads/因子挖掘/raw_data"
    output_dir = "/Users/wook/Downloads/因子挖掘/Sample data/原数据1s"
    
    # 方法1: 批量处理所有订单簿文件
    print("开始批量处理所有订单簿文件...")
    process_book_files_to_h5(raw_data_dir, output_dir)
    
    # 方法2: 处理单个文件（示例）
    # book_file = "/Users/wook/Downloads/因子挖掘/raw_data/BTCUSDT_book_snapshot_25_20250506.csv.gz"
    # process_single_book_file(book_file, output_dir)
    
    # 方法3: 检查数据结构（用于调试）
    # book_files = find_book_files(raw_data_dir)
    # if book_files:
    #     inspect_book_data(book_files[0])

开始批量处理所有订单簿文件...
找到 32 个订单簿文件待处理


                                                                                        
                                                                                        
处理: BTCUSDT_book_snapshot_25_20250506.csv.gz:   0%|          | 0/32 [00:08<?, ?文件/s]

原始数据形状: (1497122, 104)
时间范围: binance-futures - binance-futures


                                                                                        
处理: BTCUSDT_book_snapshot_25_20250506.csv.gz:   0%|          | 0/32 [00:09<?, ?文件/s]

降频后数据形状: (86393, 104)


...

原始数据形状: (1504829, 104)
时间范围: binance-futures - binance-futures


                                                                                                 
处理: BTCUSDT_book_snapshot_25_20250606.csv.gz:  97%|█████████▋| 31/32 [09:28<00:17, 17.28s/文件]

降频后数据形状: (86395, 104)


处理: BTCUSDT_book_snapshot_25_20250606.csv.gz: 100%|██████████| 32/32 [09:36<00:00, 18.02s/文件]    

✓ 已完成: BTCUSDT_book_snapshot_25_20250606.csv.gz
  保存至: /Users/wook/Downloads/因子挖掘/Sample data/原数据1s/BTCUSDT_book_snapshot_25_20250606_1s.h5
  数据压缩比: 1504829 -> 86395 行

🎉 批量处理完成!
